In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [2]:
%pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [3]:
import dagshub
dagshub.init(repo_owner='xlrboi', repo_name='delievery_time_prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f523eb5c-30c9-4fc0-acf3-da187f87ef20&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e3d2cc698c64728dfcfa41f2541f1e98aa449960c5a78e822eabe210c1b0b013




Accessing as xlrboi

Initialized MLflow to track repo "xlrboi/delievery_time_prediction"

Repository xlrboi/delievery_time_prediction initialized!

In [4]:
import mlflow

In [5]:
mlflow.set_tracking_uri("https://dagshub.com/xlrboi/delievery_time_prediction.mlflow")

In [6]:
mlflow.set_experiment("1 - Model Selection")

<Experiment: artifact_location='mlflow-artifacts:/871e052d06454db79fc046b1471cadd9', creation_time=1784386343127, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784386343127, lifecycle_stage='active', name='1 - Model Selection', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [7]:
from sklearn import set_config

set_config(transform_output="pandas")

In [8]:
df = pd.read_csv('swiggy.csv')
df

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,11:45:00,conditions Windy,High,1,Meal,motorcycle,0,No,Metropolitian,(min) 32
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,20:10:00,conditions Windy,Jam,0,Buffet,motorcycle,1,No,Metropolitian,(min) 36
45590,0x4f8d,CHENRES08DEL03,30,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,23:50:00,00:05:00,conditions Cloudy,Low,1,Drinks,scooter,0,No,Metropolitian,(min) 16
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,13:40:00,conditions Cloudy,High,0,Snack,motorcycle,1,No,Metropolitian,(min) 26


In [9]:
import numpy as np
import pandas as pd


columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"]


def change_column_names(data: pd.DataFrame):
    return (
        data.rename(str.lower,axis=1)
        .rename({
            "delivery_person_id" : "rider_id",
            "delivery_person_age": "age",
            "delivery_person_ratings": "ratings",
            "delivery_location_latitude": "delivery_latitude",
            "delivery_location_longitude": "delivery_longitude",
            "time_orderd": "order_time",
            "time_order_picked": "order_picked_time",
            "weatherconditions": "weather",
            "road_traffic_density": "traffic",
            "city": "city_type",
            "time_taken(min)": "time_taken"},
            axis=1)
    )


def data_cleaning(data: pd.DataFrame):
    minors_data = data.loc[data['age'].astype(float) < 18]
    minor_index = minors_data.index.tolist()

    six_star_data = data.loc[data['ratings'] == "6"]
    six_star_index = six_star_data.index.tolist()

    return (
        data
        .drop(columns="id")
        .drop(index=minor_index)                     # Minor riders dropped
        .drop(index=six_star_index)                  # Six-star riders dropped
        .replace("NaN ", np.nan)
        .assign(
            # City extracted from rider id
            city_name=lambda x: x["rider_id"].str.split("RES").str.get(0),

            # Numeric columns
            age=lambda x: x["age"].astype(float),
            ratings=lambda x: x["ratings"].astype(float),

            # Absolute latitude/longitude
            restaurant_latitude=lambda x: x["restaurant_latitude"].abs(),
            restaurant_longitude=lambda x: x["restaurant_longitude"].abs(),
            delivery_latitude=lambda x: x["delivery_latitude"].abs(),
            delivery_longitude=lambda x: x["delivery_longitude"].abs(),

            # Date features
            order_date=lambda x: pd.to_datetime(
                x["order_date"], dayfirst=True
            ),
            order_day=lambda x: x["order_date"].dt.day,
            order_month=lambda x: x["order_date"].dt.month,
            order_day_of_week=lambda x: (
                x["order_date"].dt.day_name().str.lower()
            ),
            is_weekend=lambda x: (
                x["order_date"]
                .dt.day_name()
                .isin(["Saturday", "Sunday"])
                .astype(int)
            ),

            # Time columns
            order_time=lambda x: pd.to_datetime(
                x["order_time"], format="mixed"
            ),
            order_picked_time=lambda x: pd.to_datetime(
                x["order_picked_time"], format="mixed"
            ),

            # Pickup time
            pickup_time_minutes=lambda x: (
                (x["order_picked_time"] - x["order_time"])
                .dt.seconds / 60
            ),

            # Order hour
            order_time_hour=lambda x: x["order_time"].dt.hour,

            # Time of day
            order_time_of_day=lambda x: (
                x["order_time_hour"].pipe(time_of_day)
            ),

            # Categorical columns
            weather=lambda x: (
                x["weather"]
                .str.replace("conditions ", "", regex=False)
                .str.lower()
                .replace("nan", np.nan)
            ),

            traffic=lambda x: x["traffic"].str.rstrip().str.lower(),
            type_of_order=lambda x: x["type_of_order"].str.rstrip().str.lower(),
            type_of_vehicle=lambda x: x["type_of_vehicle"].str.rstrip().str.lower(),
            festival=lambda x: x["festival"].str.rstrip().str.lower(),
            city_type=lambda x: x["city_type"].str.rstrip().str.lower(),

            # Multiple deliveries
            multiple_deliveries=lambda x: (
                x["multiple_deliveries"].astype(float)
            ),

            # Target column
            time_taken=lambda x: (
                x["time_taken"]
                .str.replace("(min)", "", regex=False)
                .str.strip()
                .astype(int)
            ),
        )
        .drop(columns=["order_time", "order_picked_time"])
    )



def clean_lat_long(data: pd.DataFrame, threshold=1):
    location_columns = ['restaurant_latitude',
                        'restaurant_longitude',
                        'delivery_latitude',
                        'delivery_longitude']

    return (
        data
        .assign(**{
            col: (
                np.where(data[col] < threshold, np.nan, data[col].values)
            )
            for col in location_columns
        })
    )


# extract day, day name, month and year
def extract_datetime_features(ser):
    date_col = pd.to_datetime(ser,dayfirst=True)

    return (
        pd.DataFrame(
            {
                "day": date_col.dt.day,
                "month": date_col.dt.month,
                "year": date_col.dt.year,
                "day_of_week": date_col.dt.day_name(),
                "is_weekend": date_col.dt.day_name().isin(["Saturday","Sunday"]).astype(int)
            }
        ))


def time_of_day(ser):

    return(
        pd.cut(ser,bins=[0,6,12,17,20,24],right=True,
               labels=["after_midnight","morning","afternoon","evening","night"])
    )


def drop_columns(data: pd.DataFrame, columns: list) -> pd.DataFrame:
    df = data.drop(columns=columns)
    return df


def calculate_haversine_distance(df):
    location_columns = ['restaurant_latitude',
                        'restaurant_longitude',
                        'delivery_latitude',
                        'delivery_longitude']

    lat1 = df[location_columns[0]]
    lon1 = df[location_columns[1]]
    lat2 = df[location_columns[2]]
    lon2 = df[location_columns[3]]

    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(
        dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2

    c = 2 * np.arcsin(np.sqrt(a))
    distance = 6371 * c

    return (
        df.assign(
            distance = distance)
    )

def create_distance_type(data: pd.DataFrame):
    return(
        data
        .assign(
                distance_type = pd.cut(data["distance"],bins=[0,5,10,15,25],
                                        right=False,labels=["short","medium","long","very_long"])
    ))


def perform_data_cleaning(data: pd.DataFrame):

    cleaned_data = (
        data
        .pipe(change_column_names)
        .pipe(data_cleaning)
        .pipe(clean_lat_long)
        .pipe(calculate_haversine_distance)
        .pipe(create_distance_type)
        .pipe(drop_columns,columns=columns_to_drop)
    )

    return cleaned_data.dropna()

In [10]:
cleaned_data = perform_data_cleaning(df)
cleaned_data

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45587,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,33,0,10.0,night,16.600272,very_long
45588,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45590,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45591,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [11]:
from pathlib import Path

output_path = Path("cleaned_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_data.to_csv(output_path, index=False)

In [12]:
df = pd.read_csv('cleaned_data.csv')
df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,33,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [13]:
df.columns

Index(['age', 'ratings', 'weather', 'traffic', 'vehicle_condition',
       'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival',
       'city_type', 'time_taken', 'is_weekend', 'pickup_time_minutes',
       'order_time_of_day', 'distance', 'distance_type'],
      dtype='object')

In [14]:
df.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
temp_df = df.copy().dropna()

In [17]:
X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37690,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,0,10.0,night,16.600272,very_long
37691,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,0,10.0,morning,1.489846,short
37692,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,0,15.0,night,4.657195,short
37693,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,0,5.0,afternoon,6.232393,medium


In [18]:
y

,time_taken
0,24
1,33
2,26
3,21
4,30
...,...
37690,33
37691,32
37692,16
37693,26


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [21]:
X_train.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [22]:
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [23]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [24]:
nominal_cat_cols

['weather',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'is_weekend',
 'order_time_of_day']

In [25]:
X_train.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [ ]:
# # features to fill values with mode

# features_to_fill_mode = ['multiple_deliveries','festival','city_type']
# features_to_fill_missing = [col for col in nominal_cat_cols if col not in features_to_fill_mode]

# features_to_fill_missing

In [ ]:
# # simple imputer to fill categorical vars with mode

# simple_imputer = ColumnTransformer(transformers=[
#     ("mode_imputer",SimpleImputer(strategy="most_frequent",add_indicator=True),features_to_fill_mode),
#     ("missing_imputer",SimpleImputer(strategy="constant",fill_value="missing",add_indicator=True),features_to_fill_missing)
# ],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

# simple_imputer

In [ ]:
# simple_imputer.fit_transform(X_train)

In [ ]:
# simple_imputer.fit_transform(X_train).isna().sum()

In [ ]:
# knn imputer

# knn_imputer = KNNImputer(n_neighbors=5)

In [26]:
traffic_order = ["low","medium","high","jam"]
distance_type_order = ["short","medium","long","very_long"]

In [27]:
for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic ['jam' 'medium' 'high' 'low']
distance_type ['medium' 'short' 'long' 'very_long']


In [28]:
preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,verbose_feature_names_out=False)


preprocessor

ColumnTransformer(n_jobs=-1, remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [29]:
processing_pipeline = Pipeline(steps=[
                                # ("simple_imputer",simple_imputer),
                                ("preprocess",preprocessor)
                                # ("knn_imputer",knn_imputer)
                            ])

processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(n_jobs=-1, remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [30]:
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

In [31]:
X_train_trans

,age,ratings,pickup_time_minutes,distance,weather_fog,weather_sandstorms,weather_stormy,weather_sunny,weather_windy,type_of_order_drinks,...,city_type_semi-urban,city_type_urban,is_weekend_1,order_time_of_day_evening,order_time_of_day_morning,order_time_of_day_night,traffic,distance_type,vehicle_condition,multiple_deliveries
7204,0.473684,0.56,1.0,0.404165,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,0,2.0
20974,1.000000,0.76,0.0,0.154044,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,1.0
28236,0.473684,0.80,0.5,0.002461,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1,0.0
21635,1.000000,0.92,1.0,0.460411,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0,1.0
30780,0.526316,0.76,0.5,0.243676,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16850,0.578947,0.92,0.5,0.451895,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,3.0,2.0,0,0.0
6265,0.052632,1.00,1.0,0.612270,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0,1,1.0
11284,0.526316,0.92,0.0,0.322877,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1,0.0
860,0.947368,0.96,0.5,0.004486,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0,1.0


In [32]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.8 MB/s eta 0:00:00


In [33]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

In [34]:
from sklearn.metrics import r2_score, mean_absolute_error

In [39]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
N_SPLITS = 5

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

def build_rf(trial):
    return RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        max_depth=trial.suggest_int("max_depth", 5, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state=RANDOM_STATE, n_jobs=-1
    )

def build_gb(trial):
    return GradientBoostingRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        random_state=RANDOM_STATE
    )

# def build_svr(trial):
#     kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly"])
#     params = {"kernel": kernel, "C": trial.suggest_float("C", 0.1, 100, log=True)}
#     if kernel == "rbf":
#         params["gamma"] = trial.suggest_float("gamma", 1e-4, 1, log=True)
#     elif kernel == "poly":
#         params["degree"] = trial.suggest_int("degree", 2, 5)
#     return SVR(**params)

# def build_knn(trial):
#     return KNeighborsRegressor(
#         n_neighbors=trial.suggest_int("n_neighbors", 2, 30),
#         weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
#         p=trial.suggest_int("p", 1, 2),
#         n_jobs=-1
#     )

def build_xgb(trial):
    return XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 8),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        objective="reg:squarederror", eval_metric="mae",
        tree_method="hist", device="cuda", random_state=RANDOM_STATE
    )

def build_lgbm(trial):
    return LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        num_leaves=trial.suggest_int("num_leaves", 20, 100),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
        random_state=RANDOM_STATE, verbose=-1
    )

MODEL_BUILDERS = {
    "RF": build_rf,
    "GB": build_gb,
    # "SVR": build_svr,
    # "KNN": build_knn,
    "XGB": build_xgb,
    "LGBM": build_lgbm,
}

In [40]:
from sklearn.model_selection import cross_validate

def objective(trial, model_name, model_builder, X, y, cv):
    with mlflow.start_run(nested=True, run_name=f"{model_name}_Trial_{trial.number}"):

        model = model_builder(trial)

        cv_results = cross_validate(
            estimator=model, X=X, y=y, cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=-1,
            return_train_score=True,
            error_score="raise"
        )

        mean_cv_mae = -cv_results["test_score"].mean()
        std_cv_mae = cv_results["test_score"].std()
        train_mae = -cv_results["train_score"].mean()
        fit_time = cv_results["fit_time"].mean()
        score_time = cv_results["score_time"].mean()

        mlflow.log_param("model", model_name)
        mlflow.log_params(model.get_params())
        mlflow.log_metric("cv_mae", mean_cv_mae)
        mlflow.log_metric("cv_std", std_cv_mae)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("fit_time", fit_time)
        mlflow.log_metric("score_time", score_time)

        return mean_cv_mae


def run_study(model_name, model_builder, X, y, n_trials=15):
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        pruner=MedianPruner()
    )

    with mlflow.start_run(run_name=f"{model_name}_Model_Selection"):
        study.optimize(
            lambda trial: objective(trial, model_name, model_builder, X, y, cv),
            n_trials=n_trials,
            n_jobs=1,
            show_progress_bar=True
        )

        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_cv_mae", study.best_value)

    return study

In [41]:
results, studies = [], {}

for model_name, model_builder in MODEL_BUILDERS.items():
    print("=" * 70)
    print(f"Running Model Selection for {model_name}")
    print("=" * 70)

    start = time.time()
    study = run_study(
        model_name=model_name,
        model_builder=model_builder,
        X=X_train_trans,
        y=y_train_pt.values.ravel(),
        n_trials=15
    )

    studies[model_name] = study
    results.append({
        "Model": model_name,
        "Best CV MAE": study.best_value,
        "Trials": len(study.trials),
        "Time (min)": round((time.time() - start) / 60, 2),
        "Best Params": study.best_params,
    })

results_df = pd.DataFrame(results).sort_values("Best CV MAE").reset_index(drop=True)
print(results_df)

top_models = results_df.head(2)
print("\nTop Models\n")
print(top_models[["Model", "Best CV MAE"]])

[I 2026-07-18 16:19:55,349] A new study created in memory with name: no-name-2bade0cc-1b4d-4a2a-bd26-6fb7c2a961f9


Running Model Selection for RF


  0%|          | 0/15 [00:00<?, ?it/s]

🏃 View run RF_Trial_0 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/25727188d5c74fbfbd825418ae4f9464
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:20:20,055] Trial 0 finished with value: 0.35203837068205146 and parameters: {'n_estimators': 175, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.35203837068205146.
🏃 View run RF_Trial_1 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/2e1b5f2537014f6d9107ef95abaaad78
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:20:32,740] Trial 1 finished with value: 0.36384597216942405 and parameters: {'n_estimators': 111, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.35203837068205146.
🏃 View run RF_Tria

[I 2026-07-18 16:25:10,919] A new study created in memory with name: no-name-82de3ba5-1069-4791-8cc8-3483780bf54a


...

🏃 View run XGB_Trial_0 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/0420b71962ee48b2991bb8887e17ce89
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:32:46,534] Trial 0 finished with value: 0.34488519870187956 and parameters: {'n_estimators': 212, 'learning_rate': 0.2536999076681772, 'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'gamma': 0.2904180608409973, 'reg_alpha': 2.1423021757741068, 'reg_lambda': 0.10129197956845731}. Best is trial 0 with value: 0.34488519870187956.
🏃 View run XGB_Trial_1 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/f493d346d8eb434bb027e903922189e7
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:33:00,942] Trial 1 finished with value: 0.3465891790471042 and parameters: {'n_estimators': 31

[I 2026-07-18 16:37:08,442] A new study created in memory with name: no-name-2dfb4d40-8445-4fb9-ae91-5636f981927f


Running Model Selection for LGBM


  0%|          | 0/15 [00:00<?, ?it/s]

🏃 View run LGBM_Trial_0 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/8cbe4ead245c45b8b68c55ced57c2608
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:37:25,730] Trial 0 finished with value: 0.35100735605000255 and parameters: {'n_estimators': 212, 'learning_rate': 0.2536999076681772, 'max_depth': 8, 'num_leaves': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608}. Best is trial 0 with value: 0.35100735605000255.
🏃 View run LGBM_Trial_1 at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1/runs/f29eac6a58b7470aa925ce1d17383090
🧪 View experiment at: https://dagshub.com/xlrboi/delievery_time_prediction.mlflow/#/experiments/1
[I 2026-07-18 16:37:45,096] Trial 1 finished with value: 0.34383713914597164 and parameters: {'n_estimators': 117, 'learning_rate': 0.19030368381735815, 'max_depth': 7, 'num_leaves': 77, 'subsample': 0.706175348

In [42]:
best_model_1 = top_models.iloc[0]["Model"]
best_model_2 = top_models.iloc[1]["Model"]

study_1 = studies[best_model_1]
study_2 = studies[best_model_2]

print(f"\nBest Model : {best_model_1}")
print(study_1.best_params)

print(f"\nSecond Best Model : {best_model_2}")
print(study_2.best_params)


Best Model : LGBM
{'n_estimators': 264, 'learning_rate': 0.01875220945578641, 'max_depth': 10, 'num_leaves': 82, 'subsample': 0.9818496824692567, 'colsample_bytree': 0.9684482051282947}

Second Best Model : XGB
{'n_estimators': 348, 'learning_rate': 0.07319435578289295, 'max_depth': 10, 'min_child_weight': 7, 'subsample': 0.8868797195348295, 'colsample_bytree': 0.8412334901270925, 'gamma': 1.3696411015361143, 'reg_alpha': 0.1136250628143983, 'reg_lambda': 0.3346843561591919}


In [48]:
optuna.visualization.plot_optimization_history(study)

In [49]:
optuna.visualization.plot_param_importances(study)

In [50]:
optuna.visualization.plot_slice(study)

In [52]:
# train the model on best parameters

lgbm = LGBMRegressor(**study_1.best_params)

lgbm.fit(X_train_trans,y_train_pt.values.ravel())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004308 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 352
[LightGBM] [Info] Number of data points in the train set: 30156, number of used features: 25
[LightGBM] [Info] Start training from score -0.000000


LGBMRegressor(colsample_bytree=0.9684482051282947,
              learning_rate=0.01875220945578641, max_depth=10, n_estimators=264,
              num_leaves=82, subsample=0.9818496824692567)

In [53]:
# get the predictions
y_pred_train = lgbm.predict(X_train_trans)
y_pred_test = lgbm.predict(X_test_trans)

In [54]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [55]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

The train error is 2.91 minutes
The test error is 3.00 minutes


In [56]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")

The train r2 score is 0.85
The test r2 score is 0.84


In [57]:
xgb = XGBRegressor(**study_2.best_params)
xgb.fit(X_train_trans,y_train_pt.values.ravel())

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8412334901270925, device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=1.3696411015361143, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.07319435578289295,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=7, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=348, n_jobs=None,
             num_parallel_tree=None, ...)

In [58]:
# get the predictions
y_pred_train = xgb.predict(X_train_trans)
y_pred_test = xgb.predict(X_test_trans)

In [59]:
# get the actual predictions values

y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

In [60]:
from sklearn.metrics import mean_absolute_error, r2_score

print(f"The train error is {mean_absolute_error(y_train,y_pred_train_org):.2f} minutes")
print(f"The test error is {mean_absolute_error(y_test,y_pred_test_org):.2f} minutes")

The train error is 2.95 minutes
The test error is 3.00 minutes


In [61]:
print(f"The train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"The test r2 score is {r2_score(y_test,y_pred_test_org):.2f}")

The train r2 score is 0.85
The test r2 score is 0.84


In [62]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.344885,2026-07-18 16:32:38.960940,2026-07-18 16:32:46.534268,0 days 00:00:07.573328,0.746798,0.290418,0.253700,8,5,212,2.142302,0.101292,0.746806,COMPLETE
1,1,0.346589,2026-07-18 16:32:46.537019,2026-07-18 16:33:00.941998,0 days 00:00:14.404979,0.754547,0.917023,0.010725,10,7,313,0.003321,0.042052,0.763702,COMPLETE
2,2,0.355216,2026-07-18 16:33:00.944326,2026-07-18 16:33:13.164785,0 days 00:00:12.220459,0.809909,2.280350,0.026927,7,2,230,0.843101,0.000996,0.787643,COMPLETE
3,3,0.408396,2026-07-18 16:33:13.167207,2026-07-18 16:33:36.994624,0 days 00:00:23.827417,0.719515,4.744428,0.075001,3,5,254,6.732249,1.101506,0.751157,COMPLETE
4,4,0.362149,2026-07-18 16:33:36.997113,2026-07-18 16:33:57.162817,0 days 00:00:20.165704,0.848553,0.171943,0.013940,8,4,191,3.520481,0.001967,0.736611,COMPLETE
5,5,0.360114,2026-07-18 16:33:57.165932,2026-07-18 16:34:13.286382,0 days 00:00:16.120450,0.990875,3.875664,0.028869,7,5,299,4.983044,2.979454,0.755456,COMPLETE
6,6,0.392169,2026-07-18 16:34:13.289452,2026-07-18 16:34:32.969806,0 days 00:00:19.680354,0.797599,1.943386,0.229996,3,2,279,0.002274,1.392155,0.713568,COMPLETE
7,7,0.364783,2026-07-18 16:34:32.973134,2026-07-18 16:34:53.222361,0 days 00:00:20.249227,0.722365,4.934435,0.026000,7,2,207,0.726480,0.000985,0.940659,COMPLETE
8,8,0.350562,2026-07-18 16:34:53.224871,2026-07-18 16:35:09.229483,0 days 00:00:16.004612,0.722213,1.792329,0.160153,8,6,101,0.000380,2.067841,0.931381,COMPLETE
9,9,0.404668,2026-07-18 16:35:09.233209,2026-07-18 16:35:33.221561,0 days 00:00:23.988352,0.918882,3.187787,0.030816,3,3,287,2.729378,0.022965,0.797555,COMPLETE


In [65]:
from sklearn.compose import TransformedTargetRegressor

model = TransformedTargetRegressor(regressor=lgbm,
                                    transformer=pt)

In [66]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

scores

array([-3.07193301, -3.03610226, -3.03909789, -3.04488778, -3.02115415])

In [67]:
- scores.mean()

np.float64(3.0426350196223058)

In [70]:
from sklearn.compose import TransformedTargetRegressor

model = TransformedTargetRegressor(regressor=xgb,
                                    transformer=pt)

In [72]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

scores

array([-3.08031201, -3.05287075, -3.05215406, -3.05559754, -3.04452062])

In [73]:
- scores.mean()

np.float64(3.057090997695923)